In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/deeppythonist/brain-tumor-mri-dataset")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.models import Sequential


In [ ]:
test=keras.utils.image_dataset_from_directory(
    directory='/content/brain-tumor-mri-dataset/test',
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=32,
    image_size=(256, 256)
)

train=keras.utils.image_dataset_from_directory(
    directory='/content/brain-tumor-mri-dataset/train',
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=32,
    image_size=(256, 256)
)

In [ ]:
# Normalize
def process(image,label):
    image=tf.cast(image/255. ,tf.float32)
    return image,label

train=train.map(process)
test=test.map(process)

In [ ]:
model = Sequential()

model.add(Conv2D(16,(3,3),activation='relu',
                 padding='same',
                 input_shape=(256,256,3)))
model.add(MaxPooling2D())

model.add(Conv2D(32,(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D())



model.add(Conv2D(64,(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D())


model.add(Conv2D(128,(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D())

model.add(Conv2D(256,(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D())

model.add(Conv2D(512,(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D())

model.add(Flatten())

model.add(Dense(256,activation='relu'))
model.add(Dense(4,activation='softmax'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.fit(train,epochs=10,validation_data=test)

In [ ]:
import numpy as np

predictions = model.predict(test)
predicted_classes = np.argmax(predictions, axis=1)

print(predicted_classes)

In [ ]:
import os

classes = sorted(os.listdir('/content/brain-tumor-mri-dataset/train'))
print(classes)

In [ ]:
import cv2
import numpy as np

# Load image
img = cv2.imread('/content/CC7C8857-F56B-4524-8E66-A5DD0D03B16F-1200x1200-cropped.jpeg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (256, 256))

# Normalize
img = img.astype('float32') / 255.0

# Add batch dimension
img = np.expand_dims(img, axis=0)

# Predict
pred = model.predict(img)

# Get predicted class index
pred_idx = np.argmax(pred)

# Get class name
class_names = sorted(os.listdir('/content/brain-tumor-mri-dataset/train'))

print("Predicted Index:", pred_idx)
print("Predicted Class:", class_names[pred_idx])
print("Confidence:", pred[0][pred_idx] * 100, "%")